# Polytomous IRT Models

This notebook demonstrates the polytomous IRT models supported by the library:
- **PCM** (Partial Credit Model) - Rasch-like for ordered categories
- **RSM** (Rating Scale Model) - Same rating scale across items
- **GRM** (Graded Response Model) - Cumulative probability model
- **GPCM** (Generalized Partial Credit Model) - PCM with discrimination
- **NRM** (Nominal Response Model) - For unordered categories

Use polytomous models when your data has Likert scales, partial credit, or multiple-choice items with more than two response options.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from irt import fit

np.random.seed(42)

## 1. Data Preparation

Polytomous data: integer responses 0, 1, 2, ..., m-1 per item. Missing = NaN.

In [ ]:
# Simulate Likert-scale data: 200 persons, 8 items, 5 categories (0-4)
X = np.random.randint(0, 5, size=(200, 8)).astype(float)
# Add some missing
mask_miss = np.random.random(X.shape) < 0.05
X[mask_miss] = np.nan

print("Shape:", X.shape)
print("Categories per item (inferred):", np.nanmax(X, axis=0).astype(int) + 1)

## 2. Model Fitting

Fit each polytomous model with `fit(X, model="pcm")` etc.

In [ ]:
result_pcm = fit(X, model="pcm")
print("PCM:", result_pcm.converged, "loglik =", result_pcm.loglik)

result_gpcm = fit(X, model="gpcm")
print("GPCM:", result_gpcm.converged, "loglik =", result_gpcm.loglik)

result_grm = fit(X, model="grm")
print("GRM:", result_grm.converged, "loglik =", result_grm.loglik)

## 3. Parameter Interpretation

Each model has different parameter structures. Use `item_report()` and `params`.

In [ ]:
print("PCM params:", result_pcm.params.keys())
print("PCM b (step params) shape:", result_pcm.params["b"].shape)
print(result_pcm.item_report().head())

## 4. Scoring

EAP, MAP, and MLE for polytomous models.

In [ ]:
scores_eap = result_pcm.score(method="eap")
scores_map = result_pcm.score(method="map")
print("EAP theta[:5]:", scores_eap.theta[:5])
print("EAP SE[:5]:", scores_eap.se[:5])

## 5. Category Characteristic Curves

Plot P(X=c|theta) vs theta for each category.

In [ ]:
fig, ax = result_pcm.plot_icc(items=[0])
ax.set_title("Item 0: Category Characteristic Curves (PCM)")
plt.tight_layout()
plt.show()

## 6. Model Comparison

Compare AIC/BIC across models.

In [ ]:
for model_name, res in [("PCM", result_pcm), ("GPCM", result_gpcm), ("GRM", result_grm)]:
    mf = res.model_fit()
    print(f"{model_name}: AIC={mf['aic']:.1f}, BIC={mf['bic']:.1f}")

## 7. Diagnostics

Item fit, person fit, and model fit for polytomous.

In [ ]:
item_fit = result_pcm.item_fit()
print(item_fit[['item', 'n_obs', 'infit_ms', 'outfit_ms']].head())

person_fit = result_pcm.person_fit()
print(person_fit[['person', 'theta', 'infit_ms']].head())

## 8. RSM and NRM

RSM assumes the same rating scale across items. NRM is for nominal (unordered) categories.

In [ ]:
result_rsm = fit(X, model="rsm")
print("RSM converged:", result_rsm.converged)
print("RSM tau (common steps):", result_rsm.params["tau"])

result_nrm = fit(X, model="nrm")
print("NRM converged:", result_nrm.converged)